### Importazione di moduli e pacchetti

In [36]:
import csv
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display
import seaborn as sn  #Per heatmap
import time
import scipy as sp
import os
import librosa as lb

#sklearn
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, LabelEncoder,OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV, ParameterGrid, train_test_split
from sklearn.ensemble import RandomForestRegressor

In [37]:
data = pd.read_csv('../data/development.csv', index_col=0)
data['path'] = data['path'].map(lambda x: x.split('/')[1])
display(data)

,sampling_rate,age,gender,ethnicity,mean_pitch,max_pitch,min_pitch,jitter,shimmer,energy,zcr_mean,spectral_centroid_mean,tempo,hnr,num_words,num_characters,num_pauses,silence_duration,path
Id,,,,,,,,,,,,,,,,,,,
0,22050,24.0,female,arabic,1821.69060,3999.7170,145.43066,0.013795,0.082725,0.002254,0.210093,3112.257251,[151.99908088],-123.999726,69,281,39,23.846893,1.wav
1,22050,22.5,female,hungarian,1297.81870,3998.8590,145.37268,0.025349,0.096242,0.007819,0.078849,1688.016389,[129.19921875],-86.928478,69,281,21,19.388662,2.wav
2,22050,22.0,female,portuguese,1332.85240,3998.8025,145.42395,0.019067,0.119456,0.002974,0.105365,2576.901706,[117.45383523],-98.450670,69,281,1,21.640998,3.wav
3,22050,22.0,female,english,1430.34990,3998.4510,147.98083,0.017004,0.102389,0.022371,0.173701,3269.751413,[117.45383523],-56.459762,69,281,9,19.644127,4.wav
4,22050,22.0,male,dutch,1688.72340,3998.6113,145.44772,0.028027,0.124831,0.005369,0.107279,1930.897375,[112.34714674],-80.349204,69,281,11,18.041905,5.wav
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2928,22050,24.0,male,english,1641.14930,3999.1616,145.39359,0.023647,0.115361,0.001879,0.111799,2188.853478,[184.5703125],-100.921055,69,281,11,17.461406,2929.wav
2929,22050,15.0,female,igbo,1089.60050,3984.6550,145.58409,0.015317,0.126740,0.000339,0.070508,2712.362323,[83.35433468],6.757283,0,0,1,1.509206,2930.wav
2930,22050,17.0,female,igbo,994.46484,3989.1785,148.97475,0.009677,0.103535,0.001464,0.058442,2248.698477,[89.10290948],-53.913449,1,9,1,1.645034,2931.wav


## Data extraction - Functions

In [38]:
def load_data(audio_files, folder_path, sample_rate):
    time_stamps = []
    audio_arrays = {}
    mfccs = {}
    for file_name in audio_files:
        if file_name.endswith(".wav"):
            audio_array = lb.load(folder_path + file_name)
            trimmed_audio_array= lb.effects.trim(audio_array[0])
            time_stamps.append(len(trimmed_audio_array[0]))
            audio_arrays[file_name] = trimmed_audio_array[0]
            mfccs[file_name] = lb.feature.mfcc(y=trimmed_audio_array[0], sr=sample_rate, n_mfcc=15, n_fft=2048,hop_length=512)
    return (audio_arrays, mfccs, np.array(time_stamps))

In [39]:
def standardize_time_mfcc(mfcc,num_buckets):
    reduced_list =[]
    existing_int = mfcc.shape[1]
    for i in range(0,existing_int-(existing_int%num_buckets),existing_int//num_buckets):
        reduced_list.append(np.mean(mfcc[:, i:i+(mfcc.shape[1]//num_buckets)], axis=1).flatten())
    return np.array(reduced_list).transpose()

In [40]:
def standardize_all(mfcc_dict,num_buckets):
    standard_mfccs = {}
    for name, mfcc in mfcc_dict.items():
        standard_mfccs[name] = standardize_time_mfcc(mfcc,num_buckets)
    return standard_mfccs

In [41]:
def reshape_all(mfcc_dict):
    reshaped_mfccs = {}
    for name, mfcc in mfcc_dict.items():
        reshaped_mfccs[name] = mfcc.flatten()
    return pd.DataFrame(reshaped_mfccs).T

## Training data preparation

In [42]:
SAMPLE_RATE = 22050
num_time_buckets = 20

In [43]:
audio_files = os.listdir('../data/audios_development')
audio_arrays,mfccs,time_sampl = load_data(audio_files, '../data/audios_development/', SAMPLE_RATE)

In [44]:
standardized_mfccs = standardize_all(mfccs, num_time_buckets)
reshaped_mfccs = reshape_all(standardized_mfccs)
display(reshaped_mfccs)

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
1.wav,-423.020538,-332.887329,-361.605164,-347.482208,-389.083527,-328.815033,-398.173309,-336.800690,-379.510284,-345.724213,...,-1.942417,-5.246557,-4.376464,0.120627,-5.158453,-1.831522,-3.805745,-3.881568,-4.000273,-8.090446
10.wav,-464.710785,-267.225800,-201.898865,-216.123856,-173.217758,-220.823898,-91.316536,-163.099533,-190.114258,-243.981277,...,4.086848,8.919844,12.513921,-1.471938,-5.042432,-2.831152,-1.790807,0.321547,1.039343,-6.841827
100.wav,-318.136597,-421.639404,-281.229004,-350.477203,-380.836121,-313.451111,-382.842468,-272.421265,-430.260315,-280.430054,...,-0.412136,2.075336,6.061966,6.569131,4.929160,3.960616,9.124262,10.854587,7.318303,9.405965
1000.wav,-396.088837,-338.936218,-441.260010,-421.752289,-436.263184,-393.596039,-496.009613,-445.108246,-517.327332,-557.111511,...,-0.204290,-18.454632,-7.708149,2.199664,-3.484957,-9.490080,-12.075005,-3.242330,-2.788409,-8.915504
1001.wav,-399.899261,-363.629517,-339.692078,-331.780243,-292.952393,-279.624115,-329.839478,-323.489655,-297.451721,-304.272430,...,-0.337629,-3.771065,-12.894007,-20.165813,-24.442867,-14.174511,-6.639238,-14.892796,-18.004873,-17.359259
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995.wav,-237.101822,-251.029114,-225.179825,-240.902634,-214.896286,-328.902344,-288.886658,-309.547485,-216.283569,-249.384491,...,-0.380812,5.354548,-7.917774,-6.913934,-12.418830,-6.988984,0.467288,-3.999441,4.249442,-0.261314
996.wav,-260.329346,-303.837067,-268.231964,-296.425598,-339.485199,-265.379089,-306.549103,-267.765625,-334.914368,-252.238235,...,1.907920,4.307010,6.263086,4.819046,6.840928,0.700918,4.721804,7.176890,6.416972,4.897392
997.wav,-253.112915,-137.904434,-214.022568,-279.082306,-233.791412,-256.458740,-281.646332,-233.430222,-238.886230,-231.248749,...,-15.474440,-9.980843,-12.830086,-9.290381,-5.878903,-2.367623,-8.148173,-0.053261,0.816683,-4.935268
998.wav,-396.649658,-330.637207,-297.137207,-334.702545,-309.985260,-382.760620,-281.101410,-342.125122,-270.877258,-336.018799,...,-7.688023,-12.178954,-8.236415,-5.629763,-8.098529,-6.090665,-4.651963,-11.114296,-4.836049,-4.849733


### Ethnicity subgroups identification

In [45]:
sentence_feat = data.loc[:,['num_words','num_characters']].values
uniq, ind, count= np.unique(np.array(sentence_feat), return_counts=True, return_inverse=True, axis=0)
display(count)
print(uniq[-1])

array([ 409,    2,    1,   39,    8,    5,    2,    3,    3,    3,    1,
          8,    5,   19,    9,   16,    7,    7,    1,    4,    1,    2,
          9,    8,   27,   66,   26,    6,    5,    7,    2,   58,    1,
         52,   56,    6,   51,    5,    9,    4,    8,    4,    1,    2,
          3,    3,   15,    1,    6,    2,   28,    3,   55,   28,    3,
          1,   44,    1,    1,    5,    4,    5,    1,    1,    1,    2,
          6,    3,   11,   16,    1,    1,    1,    1,    1,    1, 1710])

[ 69 281]


In [46]:
data.loc[ind==76,'ethnicity'] = 1
data.loc[ind!=76,'ethnicity'] = 0
clean_data = data.drop(columns=['num_words','num_characters','num_pauses','sampling_rate','max_pitch','min_pitch'])
clean_data['tempo'] = clean_data['tempo'].map(lambda x: float(x[1:-1]))
display(clean_data)

,age,gender,ethnicity,mean_pitch,jitter,shimmer,energy,zcr_mean,spectral_centroid_mean,tempo,hnr,silence_duration,path
Id,,,,,,,,,,,,,
0,24.0,female,1,1821.69060,0.013795,0.082725,0.002254,0.210093,3112.257251,151.999081,-123.999726,23.846893,1.wav
1,22.5,female,1,1297.81870,0.025349,0.096242,0.007819,0.078849,1688.016389,129.199219,-86.928478,19.388662,2.wav
2,22.0,female,1,1332.85240,0.019067,0.119456,0.002974,0.105365,2576.901706,117.453835,-98.450670,21.640998,3.wav
3,22.0,female,1,1430.34990,0.017004,0.102389,0.022371,0.173701,3269.751413,117.453835,-56.459762,19.644127,4.wav
4,22.0,male,1,1688.72340,0.028027,0.124831,0.005369,0.107279,1930.897375,112.347147,-80.349204,18.041905,5.wav
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2928,24.0,male,1,1641.14930,0.023647,0.115361,0.001879,0.111799,2188.853478,184.570312,-100.921055,17.461406,2929.wav
2929,15.0,female,0,1089.60050,0.015317,0.126740,0.000339,0.070508,2712.362323,83.354335,6.757283,1.509206,2930.wav
2930,17.0,female,0,994.46484,0.009677,0.103535,0.001464,0.058442,2248.698477,89.102909,-53.913449,1.645034,2931.wav


### Joining with MFCCs

In [47]:
joined_data = clean_data.join(reshaped_mfccs, on='path').drop(columns=['path'])
display(joined_data)

,age,gender,ethnicity,mean_pitch,jitter,shimmer,energy,zcr_mean,spectral_centroid_mean,tempo,...,290,291,292,293,294,295,296,297,298,299
Id,,,,,,,,,,,,,,,,,,,,,
0,24.0,female,1,1821.69060,0.013795,0.082725,0.002254,0.210093,3112.257251,151.999081,...,-1.942417,-5.246557,-4.376464,0.120627,-5.158453,-1.831522,-3.805745,-3.881568,-4.000273,-8.090446
1,22.5,female,1,1297.81870,0.025349,0.096242,0.007819,0.078849,1688.016389,129.199219,...,1.602238,-4.590589,1.433791,0.820234,2.581253,2.410373,-4.295688,4.829690,-2.975813,0.896594
2,22.0,female,1,1332.85240,0.019067,0.119456,0.002974,0.105365,2576.901706,117.453835,...,3.076923,-4.109672,-0.462944,1.148665,2.150095,-0.915860,-1.507109,2.374164,4.178719,3.131405
3,22.0,female,1,1430.34990,0.017004,0.102389,0.022371,0.173701,3269.751413,117.453835,...,-5.853500,-16.481180,-12.324727,-15.200755,-9.736434,-11.306305,-17.313793,-11.241422,-13.679470,-18.686422
4,22.0,male,1,1688.72340,0.028027,0.124831,0.005369,0.107279,1930.897375,112.347147,...,6.182858,-0.351926,5.084396,8.022604,6.370649,2.164866,6.037231,3.468850,5.418577,9.559019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2928,24.0,male,1,1641.14930,0.023647,0.115361,0.001879,0.111799,2188.853478,184.570312,...,2.097236,-0.667882,2.911166,3.991309,3.089388,0.143708,1.004856,-3.498344,0.469718,0.205209
2929,15.0,female,0,1089.60050,0.015317,0.126740,0.000339,0.070508,2712.362323,83.354335,...,-1.984530,4.366840,-2.148508,-1.667701,2.311760,1.714393,6.971056,3.783729,-4.724131,-4.516420
2930,17.0,female,0,994.46484,0.009677,0.103535,0.001464,0.058442,2248.698477,89.102909,...,-4.999870,-11.469429,-10.364605,-11.625636,-0.410724,-13.186292,-10.333710,-2.208193,-1.970908,-3.222235


### Splitting into males and females

In [48]:
display(reshaped_mfccs)

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
1.wav,-423.020538,-332.887329,-361.605164,-347.482208,-389.083527,-328.815033,-398.173309,-336.800690,-379.510284,-345.724213,...,-1.942417,-5.246557,-4.376464,0.120627,-5.158453,-1.831522,-3.805745,-3.881568,-4.000273,-8.090446
10.wav,-464.710785,-267.225800,-201.898865,-216.123856,-173.217758,-220.823898,-91.316536,-163.099533,-190.114258,-243.981277,...,4.086848,8.919844,12.513921,-1.471938,-5.042432,-2.831152,-1.790807,0.321547,1.039343,-6.841827
100.wav,-318.136597,-421.639404,-281.229004,-350.477203,-380.836121,-313.451111,-382.842468,-272.421265,-430.260315,-280.430054,...,-0.412136,2.075336,6.061966,6.569131,4.929160,3.960616,9.124262,10.854587,7.318303,9.405965
1000.wav,-396.088837,-338.936218,-441.260010,-421.752289,-436.263184,-393.596039,-496.009613,-445.108246,-517.327332,-557.111511,...,-0.204290,-18.454632,-7.708149,2.199664,-3.484957,-9.490080,-12.075005,-3.242330,-2.788409,-8.915504
1001.wav,-399.899261,-363.629517,-339.692078,-331.780243,-292.952393,-279.624115,-329.839478,-323.489655,-297.451721,-304.272430,...,-0.337629,-3.771065,-12.894007,-20.165813,-24.442867,-14.174511,-6.639238,-14.892796,-18.004873,-17.359259
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995.wav,-237.101822,-251.029114,-225.179825,-240.902634,-214.896286,-328.902344,-288.886658,-309.547485,-216.283569,-249.384491,...,-0.380812,5.354548,-7.917774,-6.913934,-12.418830,-6.988984,0.467288,-3.999441,4.249442,-0.261314
996.wav,-260.329346,-303.837067,-268.231964,-296.425598,-339.485199,-265.379089,-306.549103,-267.765625,-334.914368,-252.238235,...,1.907920,4.307010,6.263086,4.819046,6.840928,0.700918,4.721804,7.176890,6.416972,4.897392
997.wav,-253.112915,-137.904434,-214.022568,-279.082306,-233.791412,-256.458740,-281.646332,-233.430222,-238.886230,-231.248749,...,-15.474440,-9.980843,-12.830086,-9.290381,-5.878903,-2.367623,-8.148173,-0.053261,0.816683,-4.935268
998.wav,-396.649658,-330.637207,-297.137207,-334.702545,-309.985260,-382.760620,-281.101410,-342.125122,-270.877258,-336.018799,...,-7.688023,-12.178954,-8.236415,-5.629763,-8.098529,-6.090665,-4.651963,-11.114296,-4.836049,-4.849733


In [49]:
female_full_data = joined_data[joined_data['gender']=='female'].drop(columns=['gender'])
male_full_data = joined_data[joined_data['gender']=='male'].drop(columns=['gender'])

female = female_full_data.drop(columns=['age'])
female.columns = female.columns.astype(str)
female_age = female_full_data['age']

male = male_full_data.drop(columns=['age'])
male.columns = male.columns.astype(str)
male_age = male_full_data['age']

## Test data preparation

In [50]:
data_evaluation = pd.read_csv('../data/evaluation.csv', index_col=0)
data_evaluation['path'] = data_evaluation['path'].map(lambda x: x.split('/')[1])
display(data_evaluation)

,sampling_rate,gender,ethnicity,mean_pitch,max_pitch,min_pitch,jitter,shimmer,energy,zcr_mean,spectral_centroid_mean,tempo,hnr,num_words,num_characters,num_pauses,silence_duration,path
Id,,,,,,,,,,,,,,,,,,
0,22050,male,spanish,1056.91740,3945.1610,145.38750,0.022082,0.171076,0.003136,0.032963,1549.607050,[80.74951172],-116.662338,69,281,2,38.198503,1.wav
1,22050,male,xiang,1231.84570,3999.1720,145.56432,0.026571,0.132585,0.006783,0.123895,2344.817369,[89.10290948],-78.253824,69,281,27,29.605442,2.wav
2,22050,male,igbo,958.29065,3445.4490,145.67374,0.018044,0.096289,0.004478,0.089149,1939.574896,[123.046875],-71.630742,6,22,2,2.275556,3.wav
3,22050,female,spanish,1396.54170,3998.8948,145.41223,0.027290,0.088901,0.014893,0.097054,1832.059113,[123.046875],-101.533013,69,281,31,22.151837,4.wav
4,22050,male,spanish,1633.86770,3999.7632,145.36313,0.021621,0.103855,0.001369,0.140950,2534.611168,[112.34714674],-134.914070,69,281,35,22.430476,5.wav
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
686,22050,male,igbo,570.62740,3900.6730,145.67577,0.018842,0.079197,0.004545,0.072083,1946.502158,[112.34714674],-42.895295,6,15,1,1.861950,687.wav
687,22050,male,igbo,974.13965,3919.0024,145.90408,0.024367,0.117492,0.000878,0.076900,3319.620800,[112.34714674],-144.881089,7,22,1,4.876190,688.wav
688,22050,female,serbian,1113.27650,3999.3510,145.38307,0.020637,0.089355,0.009148,0.095613,1973.127197,[112.34714674],-73.559944,69,281,15,25.541950,689.wav


In [51]:
display(data_evaluation.loc[454,:])

sampling_rate                      22050
gender                            female
ethnicity                      taiwanese
mean_pitch                     1051.0619
max_pitch                      3997.4036
min_pitch                      145.44014
jitter                          0.022114
shimmer                         0.084117
energy                          0.003421
zcr_mean                        0.106298
spectral_centroid_mean       2073.085801
tempo                     [151.99908088]
hnr                          -108.084637
num_words                             69
num_characters                       281
num_pauses                            46
silence_duration               29.466122
path                             455.wav
Name: 454, dtype: object

In [52]:
np.unique(data_evaluation['sampling_rate'].values)

array([22050])

In [53]:
data_evaluation.loc[data_evaluation['num_characters']==281,'ethnicity'] = 1 # and data_evaluation['num_words']==69),'ethnicity'] = 1
data_evaluation.loc[data_evaluation['num_characters']!=281,'ethnicity'] = 0 # or data_evaluation['num_words']!=69),'ethnicity'] = 0
clean_data_evaluation = data_evaluation.drop(columns=['num_words','num_characters','num_pauses','sampling_rate','max_pitch','min_pitch'])
clean_data_evaluation['tempo'] = clean_data_evaluation['tempo'].map(lambda x: float(x[1:-1]))
display(clean_data_evaluation)

,gender,ethnicity,mean_pitch,jitter,shimmer,energy,zcr_mean,spectral_centroid_mean,tempo,hnr,silence_duration,path
Id,,,,,,,,,,,,
0,male,1,1056.91740,0.022082,0.171076,0.003136,0.032963,1549.607050,80.749512,-116.662338,38.198503,1.wav
1,male,1,1231.84570,0.026571,0.132585,0.006783,0.123895,2344.817369,89.102909,-78.253824,29.605442,2.wav
2,male,0,958.29065,0.018044,0.096289,0.004478,0.089149,1939.574896,123.046875,-71.630742,2.275556,3.wav
3,female,1,1396.54170,0.027290,0.088901,0.014893,0.097054,1832.059113,123.046875,-101.533013,22.151837,4.wav
4,male,1,1633.86770,0.021621,0.103855,0.001369,0.140950,2534.611168,112.347147,-134.914070,22.430476,5.wav
...,...,...,...,...,...,...,...,...,...,...,...,...
686,male,0,570.62740,0.018842,0.079197,0.004545,0.072083,1946.502158,112.347147,-42.895295,1.861950,687.wav
687,male,0,974.13965,0.024367,0.117492,0.000878,0.076900,3319.620800,112.347147,-144.881089,4.876190,688.wav
688,female,1,1113.27650,0.020637,0.089355,0.009148,0.095613,1973.127197,112.347147,-73.559944,25.541950,689.wav


In [54]:
audio_files_evaluation = os.listdir('../data/audios_evaluation')
audio_arrays_evaluation,mfccs_evaluation,time_sampl_evaluation = load_data(audio_files_evaluation, '../data/audios_evaluation/', SAMPLE_RATE)

In [55]:
standardized_mfccs_evaluation = standardize_all(mfccs_evaluation, num_time_buckets)
reshaped_mfccs_evaluation = reshape_all(standardized_mfccs_evaluation)
display(reshaped_mfccs_evaluation)

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
1.wav,-409.024384,-318.300507,-356.190277,-345.002319,-337.278931,-334.057831,-357.991608,-326.944092,-306.628845,-350.886414,...,3.988764,1.971522,4.218048,2.856964,0.434794,1.786844,4.884138,7.342025,6.462479,1.600923
10.wav,-435.284241,-392.018524,-248.948151,-323.007416,-292.441101,-270.305969,-284.768127,-329.105835,-327.972534,-353.197693,...,-5.979064,-5.135306,3.936126,0.141418,-5.740375,-13.966771,-5.430674,-2.823639,-10.169382,-2.961752
100.wav,-293.302338,-247.493042,-293.399414,-261.878693,-259.596832,-272.239227,-267.624451,-311.161499,-248.529968,-275.995758,...,14.765183,14.073303,12.891003,8.772925,11.083576,10.430563,14.053826,11.543507,11.924887,10.976063
101.wav,-193.993469,-331.810791,-270.091370,-290.040497,-326.564789,-264.394196,-321.724152,-238.821533,-321.110321,-241.872086,...,8.019660,7.337913,9.528759,6.420187,6.243932,4.817814,8.012930,5.655083,5.977262,7.757629
102.wav,-431.241058,-335.702271,-277.690338,-265.180511,-357.530884,-275.078308,-324.168732,-250.916199,-342.306335,-235.404755,...,-13.446664,-14.555564,-13.896852,-6.870616,-10.348890,-13.119678,-8.803543,-8.120754,-8.011740,-12.725560
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95.wav,-519.660828,-312.015198,-345.898254,-301.076508,-305.187042,-230.281097,-357.826508,-327.026001,-359.214020,-415.862976,...,0.992612,7.384030,1.896030,-11.600630,-2.917705,6.524923,6.019863,13.066234,7.906710,11.046993
96.wav,-335.466797,-317.808655,-165.252213,-214.704208,-170.736435,-202.559998,-216.286987,-148.046417,-180.815384,-172.436188,...,0.697124,4.973277,-6.263444,-3.027616,0.056703,2.821440,-2.695671,1.973516,3.052486,0.571430
97.wav,-503.112793,-443.052155,-453.366699,-453.388092,-230.930328,-284.111115,-296.293030,-288.297302,-323.740601,-304.597900,...,7.643392,-8.778545,-5.423125,-2.170265,-9.004732,-16.067131,-6.832082,0.283662,-1.104228,0.164042
98.wav,-261.113708,-330.292542,-284.257782,-275.578186,-321.897217,-259.656433,-314.730774,-286.421936,-298.456909,-323.157501,...,0.490386,3.241124,-0.128632,4.476760,8.151598,0.362155,2.285372,4.306463,8.140712,1.063711


In [56]:
joined_data_evaluation = clean_data_evaluation.join(reshaped_mfccs_evaluation, on='path').drop(columns=['path'])

In [57]:
display(joined_data_evaluation.loc[456,:])

gender           female
ethnicity             1
mean_pitch    1702.4655
jitter         0.014076
shimmer         0.07347
                ...    
295           -0.036455
296            3.431105
297            9.387039
298            4.055579
299           -2.855204
Name: 456, Length: 311, dtype: object

In [58]:
female_full_data_evaluation = joined_data_evaluation[joined_data_evaluation['gender']=='female'].drop(columns=['gender'])
male_full_data_evaluation = joined_data_evaluation[joined_data_evaluation['gender']=='male'].drop(columns=['gender'])
female_full_data_evaluation.columns = female_full_data_evaluation.columns.astype(str)
male_full_data_evaluation.columns = male_full_data_evaluation.columns.astype(str)

In [59]:
ypred_female = RandomForestRegressor(n_jobs=-1).fit(female, female_age).predict(female_full_data_evaluation)
ypred_male = RandomForestRegressor(n_jobs=-1).fit(male, male_age).predict(male_full_data_evaluation)

In [60]:
ypred_female_series = pd.Series(ypred_female, name='Predicted', index=female_full_data_evaluation.index)
ypred_male_series = pd.Series(ypred_male, name='Predicted', index=male_full_data_evaluation.index)
ypred = pd.concat([ypred_female_series, ypred_male_series]).sort_index()
display(ypred)

Id
0      33.190
1      30.280
2      22.560
3      31.485
4      34.330
        ...  
686    20.290
687    24.628
688    28.565
689    39.370
690    20.510
Name: Predicted, Length: 691, dtype: float64

In [61]:
display(ypred.loc[454])

np.float64(38.635)

In [62]:
pd.DataFrame(ypred, index=data_evaluation.index).to_csv('submissionMaddalena.csv')